# 02 · A reasoning chain in three calls
**12 minutes · Classify → assess → reply**

Here, Python controls three separate model calls and passes ordinary text
between them. These application stages are distinct from a model's internal
reasoning within one call.

**The setup cell** prepares a customer request and known order facts.
The stages receive only the information explicitly supplied to them.

In [ ]:
from IPython.display import display
from utils.workshop import FailOnce, Rates, Run, Settings, compare

settings = Settings.from_env()
rates = Rates.load(model_id=settings.model_id)
print(f"{settings.model_id} → {settings.resolved_api}")

order_facts = "ORDER-1042 is in_transit and four calendar days late."

## Edit the prompts

Change the customer request in `instruction` and each stage's behavior in `system_prompts`. The facts remain a separate input; intermediate text is still passed from one stage to the next.

Rerun this cell after edits, then rerun the relevant example. For comparisons, rerun both baseline and tuned versions.

In [ ]:
instruction = "My order ORDER-1042 has not arrived. What should happen next?"
system_prompts = {
    "classify": "Classify the issue in a short label.",
    "assess": "Assess the delay and suggest a next step. Do not invent policy.",
    "draft": (
        "Write a brief customer update. "
        "Do not promise a refund or delivery date."
    ),
}

## Build the chain

| Stage | Receives | Produces |
|---|---|---|
| `classify` | Customer request | A short issue label |
| `assess` | That label plus the order facts | Assessment and suggested next step |
| `draft` | Assessment text | A brief customer reply |

`.text` carries each result into the next request. The final stage depends
on what the assessment preserves, so inspect intermediate outputs when the
final answer loses a fact.

The calls run sequentially in one timed block. Each adds tokens and latency
to the completed task.

In [ ]:
def run_chain(
    label, system_prompts, instruction, order_facts,
    effort="high", before_assessment=None,
):
    with Run(label, settings, rates) as run:
        classification = run.generate(
            "classify", instruction,
            instructions=system_prompts["classify"],
            reasoning_effort=effort,
        )
        assessment = run.generate(
            "assess", f"Issue: {classification.text}\nFacts: {order_facts}",
            instructions=system_prompts["assess"],
            reasoning_effort="medium", before_attempt=before_assessment,
        )
        answer = run.generate(
            "draft", assessment.text,
            instructions=system_prompts["draft"],
            reasoning_effort="medium",
        )
    print("Classification:", classification.text)
    print("Assessment:", assessment.text)
    return run, answer.text

## Baseline · High effort for classification

`effort="high"` applies to classification; assessment and drafting already
use `"medium"`. The function prints the intermediate results and returns
the final answer with its measurements.

**Look for:** three model-call rows. Notice how much of the task's time and
token use belongs to the simple classification stage.

In [ ]:
baseline_run, baseline_answer = run_chain(
    "02_chain_baseline",
    system_prompts=system_prompts, instruction=instruction,
    order_facts=order_facts, effort="high",
)
print(baseline_answer)
display(baseline_run.metrics())
baseline_run.save()

## Recovery · Fail the middle stage once

The simulated failure happens before assessment sends its model request.
Classification is already stored in a Python variable and is reused.

**Expected trace:** classification once, assessment twice, and drafting once.
That is three logical model calls and four attempts, including one local
simulated failure. No model tokens are consumed by the injected failure.

In [ ]:
recovery_run, recovery_answer = run_chain(
    "02_chain_recovery",
    system_prompts=system_prompts, instruction=instruction,
    order_facts=order_facts, effort="high",
    before_assessment=FailOnce('assessment model request'),
)
print(recovery_answer)
display(recovery_run.metrics())
recovery_run.save()

## Tune · Lower classification effort

Change `high` to `medium` for the first stage. The other prompts, model, and
effort settings stay fixed.

The displayed comparison covers all three stages. A cheaper or faster
classification may produce only a small whole-task improvement if most work
happens later. Its wording can also change, so review the downstream answer.

In [ ]:
tuned_run, tuned_answer = run_chain(
    "02_chain_tuned",
    system_prompts=system_prompts, instruction=instruction,
    order_facts=order_facts, effort="medium",
)
print(tuned_answer)
display(compare(baseline_run, tuned_run))
tuned_run.save()

**Review both answers:** does the chain recognize a delivery delay, preserve
the four-day delay, and recommend a carrier inquiry without inventing policy?
There is no automatic quality score.

Effort is mapped for OpenAI and supported thinking-capable Claude models.
Converse leaves the separate reasoning-token count unavailable; this does
not imply zero reasoning. A simple classification may show little benefit
from changing effort. Report the observed result and rehearse repeated runs.